# Digitalify

This notebook walks through the full pipeline:

- define a small multi-layer perceptron in PyTorch
- trace the model with torch.fx to get an operator-level graph
- convert it to structural VHDL
- inspect and save the generated files
- run a functional simulation with GHDL

The network is intentionally tiny so the output VHDL is easy to read.

## Imports and setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import torch.fx as fx

import digitalify
from digitalify.exceptions import UnsupportedOperatorError

print(f'PyTorch {torch.__version__}')
print(f'digitalify loaded from: {digitalify.__file__}')

## Define the model

A two-hidden-layer MLP with ReLU activations, loosely inspired by the first
layers of a classic MNIST classifier, but kept small so the VHDL stays readable.

In [ ]:
class SimpleMLP(nn.Module):
    """Input: (batch, 16)  ->  Output: (batch, 4)"""
    def __init__(self):
        super().__init__()
        self.fc1   = nn.Linear(16, 8)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Linear(8, 4)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return x

model = SimpleMLP().eval()
print(model)

dummy = torch.zeros(1, 16)
print('Output shape:', model(dummy).shape)

## Trace with torch.fx

torch.fx.symbolic_trace converts the model into an operator-level computation
graph where every forward-pass call becomes an explicit node. We trace the
float model directly; digitalify's importer converts the weights to INT8
internally when building the IR.

If you already have a quantized model (produced by torchao or another pipeline),
pass it to symbolic_trace instead — the importer handles both.

In [ ]:
gm = fx.symbolic_trace(model)
print('=== FX graph ===')
gm.graph.print_tabular()

## Build the IR graph

The frontend importer walks the FX graph, recognises each module type, and
produces a technology-independent IRGraph. Each linear layer's weights are
rounded and clamped to INT8 at this stage, matching what the VHDL will compute.

In [ ]:
from digitalify.frontend import from_fx_graph

ir = from_fx_graph(gm, input_shape=(16,))

print(f'Graph name   : {ir.name}')
print(f'Input shape  : {ir.input_shape}')
print(f'Output shape : {ir.output_shape}')
print()
for node in ir.nodes:
    w = node.attrs.get('weight')
    info = f'weight {w.shape} dtype={w.dtype}' if w is not None else 'no weight'
    print(f'  {node.name:<20}  op={node.op_type:<18}  '
          f'in={node.input_shape}  out={node.output_shape}  {info}')

## Generate VHDL

digitalify.convert turns the IR into a dictionary of VHDL source files and
optionally writes them to disk. One file is produced per layer, plus a
top-level structural entity that wires everything together.

In [ ]:
OUTPUT_DIR = 'generated_vhdl'

vhdl_files = digitalify.convert(
    gm,
    input_shape=(16,),
    data_width=8,
    output_dir=OUTPUT_DIR,
)

print('Generated files:')
for fname in sorted(vhdl_files):
    lines = vhdl_files[fname].count('\n')
    print(f'  {fname}  ({lines} lines)')

## Inspect the generated VHDL

In [ ]:
SEP = '=' * 70

def show_vhdl(filename):
    print(SEP)
    print(f'  {filename}')
    print(SEP)
    print(vhdl_files[filename])

for fname in sorted(vhdl_files):
    show_vhdl(fname)
    print()

## Error handling: unsupported operators

digitalify raises UnsupportedOperatorError (a subclass of NotImplementedError)
whenever the graph contains a layer that has no registered VHDL handler.

In [ ]:
class ModelWithUnsupported(nn.Module):
    """Contains a BatchNorm layer, which is not yet supported."""
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 4)
        self.bn = nn.BatchNorm1d(4)

    def forward(self, x):
        return self.bn(self.fc(x))

try:
    bad_gm = fx.symbolic_trace(ModelWithUnsupported())
    digitalify.convert(bad_gm, input_shape=(4,))
except UnsupportedOperatorError as e:
    print('Caught expected error:')
    print(f'  {e}')
    print(f'  op_type   = {e.op_type}')
    print(f'  node_name = {e.node_name}')

## Functional simulation with GHDL

digitalify.simulate runs the full pipeline in a single call: it builds the IR,
generates the VHDL, computes expected outputs using a Python integer reference
model, produces a self-checking testbench, and drives it through GHDL.

The reference model replicates the exact fixed-point arithmetic of the generated
VHDL — 64-bit accumulation, arithmetic right-shift, saturation clipping — so the
simulation catches any divergence between the Python computation and the circuit.

Install GHDL before running this section:

```bash
# macOS
brew install ghdl

# Ubuntu / Debian
sudo apt install ghdl
```

In [ ]:
import shutil, subprocess

ghdl_bin = shutil.which('ghdl')
if ghdl_bin:
    ver = subprocess.check_output([ghdl_bin, '--version'], text=True).splitlines()[0]
    print(f'GHDL found: {ver}')
else:
    print('GHDL not found on PATH.')
    print('  macOS:  brew install ghdl')
    print('  Ubuntu: sudo apt install ghdl')

In [ ]:
# keep_work_dir=True preserves intermediate files for inspection
result = digitalify.simulate(
    gm,
    input_shape=(16,),
    data_width=8,
    n_vectors=8,
    seed=42,
    keep_work_dir=True,
)

print(result.summary())

### Simulation log

The log contains the full GHDL output — analysis, elaboration, and one
PASSED / FAILED report per test vector.

In [ ]:
print(result.log)

### Per-vector results

On a failure the log includes the hex value that arrived and what was expected,
which makes it straightforward to trace the divergence back to a specific layer.

In [ ]:
import re

for line in result.log.splitlines():
    if re.search(r'Test \d+:|TB SUMMARY', line):
        clean = re.sub(r'^[^:]+:[^:]+:[^:]+:[^:]+:\s*(note|error):\s*', '', line)
        print(clean)

print()
print(f'Passed: {result.n_passed}   Failed: {result.n_failed}')
print(f'Overall: {"PASS" if result.passed else "FAIL"}')

### Working directory

Because keep_work_dir was set to True, all intermediate files are preserved:
the VHDL sources, the compiled GHDL library, and the VCD waveform.

In [ ]:
print(f'Working directory: {result.work_dir}')
print()
for f in sorted(result.work_dir.iterdir()):
    print(f'  {f.name:<32}  {f.stat().st_size / 1024:6.1f} KB')

### Waveform inspection

GHDL writes a VCD file that records every signal transition. Load it into
GTKWave to visualise clk, valid_i, data_i, valid_o, and data_o.

In [ ]:
if result.vcd_path and result.vcd_path.exists():
    size_kb = result.vcd_path.stat().st_size / 1024
    print(f'VCD waveform: {result.vcd_path}  ({size_kb:.1f} KB)')
    print(f'Open with:  gtkwave {result.vcd_path}')
    print()
    for line in result.vcd_path.read_text().splitlines()[:20]:
        print(line)
else:
    print('No VCD file available.')

## Next steps

The generated VHDL files in the generated_vhdl folder can be fed directly into
any standard FPGA or ASIC synthesis flow (Vivado, Quartus, OpenLane, and others).

The library is designed to be extended. To add support for a new operator such
as Conv2d, BatchNorm, or Sigmoid:

- subclass VHDLOperator and implement the three abstract methods
- add a new entry to OPERATOR_REGISTRY in digitalify/operators/registry.py
- add a branch in from_fx_graph in digitalify/frontend/torch_importer.py to
  recognise the new module type and populate IRNode.attrs accordingly